# 01 — Not All Vectors Are Equal: Embedding Choice

**Problem:** Swapping the embedding model changes recall, latency, and memory. Picking “whatever the tutorial used” often mismatches your domain or SLA.

**In this notebook:** Encode the same corpus with two popular models, compare top-1 retrieval for a paraphrased query, and record model size / encode time.

In [ ]:
import sys
import time
from pathlib import Path

_REPO = Path.cwd().resolve()
if (_REPO / "src").is_dir():
    sys.path.insert(0, str(_REPO / "src"))

import numpy as np
from sentence_transformers import SentenceTransformer, util

docs = [
    "Refund policy: enterprise customers may request a refund within 30 days of invoice.",
    "API rate limits: standard tier allows 100 requests per minute per API key.",
    "Security: rotate API keys every 90 days and store them in a secrets manager.",
    "Billing: usage is metered monthly; overages are charged at the published rate card.",
    "Support SLAs: priority incidents receive first response within one business hour.",
]

query = "How long do I have to get my money back after purchase?"  # paraphrase of refund window

models = {
    "fast_small": "sentence-transformers/all-MiniLM-L6-v2",
    "slower_larger": "sentence-transformers/all-mpnet-base-v2",
}

results = []
for label, name in models.items():
    t0 = time.perf_counter()
    model = SentenceTransformer(name)
    load_s = time.perf_counter() - t0
    t1 = time.perf_counter()
    doc_emb = model.encode(docs, convert_to_tensor=True, show_progress_bar=False)
    q_emb = model.encode(query, convert_to_tensor=True, show_progress_bar=False)
    enc_s = time.perf_counter() - t1
    sims = util.cos_sim(q_emb, doc_emb)[0]
    top_i = int(np.argmax(sims.cpu().numpy()))
    results.append(
        {
            "label": label,
            "model": name,
            "dim": doc_emb.shape[1],
            "load_s": round(load_s, 2),
            "encode_s": round(enc_s, 4),
            "top_i": top_i,
            "top_score": float(sims[top_i]),
            "top_doc": docs[top_i][:80] + "...",
        }
    )

from tabulate import tabulate
print(tabulate([{k: v for k, v in r.items() if k != "top_doc"} for r in results], headers="keys"))
print()
for r in results:
    print(r["label"], "->", r["top_doc"])

**Takeaways**
- Smaller models are faster but can miss paraphrases that a larger model catches.
- **Measure on your data**: public benchmarks (e.g. MTEB) are a starting point, not the answer.
- **Dimension and batching** affect index size and GPU/CPU throughput in production.